# LoomGuard: unsupervised fabric defect detection

Textile mills inspect fabric for cuts, holes, stains and broken threads, mostly by eye. Labeled defect images are rare in a real factory, because defects are rare. LoomGuard learns **only from defect-free fabric** and flags anything that looks abnormal, with a heatmap showing where.

This notebook trains and compares two anomaly detectors on the texture categories of the MVTec AD benchmark (carpet, grid, leather, tile, wood):

| | **STFPM** (student–teacher) | **PatchCore** (memory bank) |
|---|---|---|
| Idea | A student network learns to imitate a pretrained teacher on normal images; it fails to imitate on defects | Store patch features of normal images; a patch far from every stored patch is a defect |
| Training | Gradient descent (student ResNet-18) | No gradients, feature extraction + coreset sampling |
| Role here | **Deployed model** (fast, exported to ONNX for CPU) | Accuracy reference |

Both models choose their decision threshold **using normal images only** (a held-out validation split), the way a factory would have to. Test labels are only used for reporting metrics.

**Before running:** Settings → Accelerator → **GPU** (T4 or P100), and **Add Input** → search for the *MVTec AD* dataset. Internet must be on (for ImageNet weights). Then *Run All*. It takes roughly 25–40 minutes for all five categories.

In [ ]:
%pip install -q onnx onnxruntime onnxscript

In [ ]:
import os, glob, json, time, random, math, shutil, platform
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_fscore_support, precision_recall_curve
import matplotlib
import matplotlib.pyplot as plt

SMOKE = os.environ.get("LOOMGUARD_SMOKE") == "1"   # local CI only: synthetic data, no downloads
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## 1. Configuration

In [ ]:
CATEGORIES      = ["carpet", "grid", "leather", "tile", "wood"]  # texture categories of MVTec AD
DEPLOY_CATEGORY = "carpet"     # the fabric model the demo app opens with
IMG_SIZE        = 256
SEED            = 42
VAL_FRACTION    = 0.10         # defect-free images held out for threshold calibration
STFPM_EPOCHS    = 100
STFPM_BATCH     = 32
STFPM_LR        = 0.4          # SGD, as in the STFPM paper
PATCHCORE_RATIO = 0.10         # coreset keeps 10% of patch features
THRESH_SIGMAS   = 3.0          # threshold = mean + 3*std of calibration (normal) scores
TOPK_FRACTION   = 0.01         # image score = mean of the hottest 1% of pixels (robust to single noisy pixels)
STFPM_RESTARTS  = 2            # train twice per category, keep the run with lower held-out loss (label-free)
CALIB_ON_DEPLOYMENT_NORMALS = True  # calibrate on half of the test-time good images (see section 7)
OUT             = "/kaggle/working/loomguard"

if SMOKE:
    CATEGORIES, DEPLOY_CATEGORY = ["carpet"], "carpet"
    IMG_SIZE, STFPM_EPOCHS, STFPM_BATCH = 64, 2, 4
    OUT = os.environ.get("LOOMGUARD_OUT", "./smoke_out")

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
seed_everything(SEED)

for sub in ["models", "figures", "samples"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

## 2. Locate the dataset
The MVTec AD copies on Kaggle use slightly different folder layouts, so the notebook searches for the category folders instead of hard-coding a path.

In [ ]:
def find_mvtec_root():
    if os.environ.get("LOOMGUARD_DATA"):
        return os.environ["LOOMGUARD_DATA"]
    for dirpath, dirnames, _ in os.walk("/kaggle/input"):
        if "carpet" in dirnames and os.path.isdir(os.path.join(dirpath, "carpet", "train", "good")):
            return dirpath
        if dirpath.count(os.sep) > 7:
            dirnames[:] = []
    raise FileNotFoundError("MVTec AD not found. Use 'Add Input' and attach an MVTec AD dataset.")

DATA_ROOT = find_mvtec_root()
CATEGORIES = [c for c in CATEGORIES if os.path.isdir(os.path.join(DATA_ROOT, c, "train", "good"))]
print("Data root:", DATA_ROOT)
print("Categories found:", CATEGORIES)

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")

def list_images(folder):
    return sorted(p for p in glob.glob(os.path.join(folder, "*")) if p.lower().endswith(IMG_EXT))

def build_split(cat):
    root = os.path.join(DATA_ROOT, cat)
    good = list_images(os.path.join(root, "train", "good"))
    rng = random.Random(SEED); rng.shuffle(good)
    n_val = max(5, int(len(good) * VAL_FRACTION))
    val, train = good[:n_val], good[n_val:]
    test = []
    for defect in sorted(os.listdir(os.path.join(root, "test"))):
        for p in list_images(os.path.join(root, "test", defect)):
            stem = os.path.splitext(os.path.basename(p))[0]
            mask = os.path.join(root, "ground_truth", defect, stem + "_mask.png")
            test.append({"path": p, "defect": defect, "label": int(defect != "good"),
                         "mask": mask if defect != "good" and os.path.exists(mask) else None})
    return train, val, test

def load_rgb(path, size=IMG_SIZE):
    img = Image.open(path).convert("RGB").resize((size, size), Image.BILINEAR)
    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)

def load_mask(path, size=IMG_SIZE):
    if path is None:
        return np.zeros((size, size), dtype=np.uint8)
    m = Image.open(path).convert("L").resize((size, size), Image.NEAREST)
    return (np.asarray(m) > 127).astype(np.uint8)

def load_stack(paths):
    return torch.stack([load_rgb(p) for p in paths])

tr, va, te = build_split(CATEGORIES[0])
print(f"{CATEGORIES[0]}: train={len(tr)} val={len(va)} test={len(te)} "
      f"(defective={sum(t['label'] for t in te)}, types={sorted({t['defect'] for t in te})})")

## 3. Models
Models take RGB images in `[0, 1]` and normalise internally, so the exported ONNX file needs no preprocessing beyond resizing.

In [ ]:
def topk_score(amap):
    # Mean of the hottest TOPK_FRACTION of pixels. A single max pixel is too noisy on fuzzy textures like carpet.
    k = max(1, int(TOPK_FRACTION * amap.shape[-1] * amap.shape[-2]))
    return amap.flatten(1).topk(k, dim=1).values.mean(1)

class Normalize(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return (x - self.mean) / self.std

class GaussianBlur(nn.Module):
    # Separable Gaussian blur written with conv2d so it exports cleanly to ONNX.
    def __init__(self, sigma=4.0):
        super().__init__()
        radius = int(4 * sigma + 0.5)
        x = torch.arange(-radius, radius + 1, dtype=torch.float32)
        k = torch.exp(-x ** 2 / (2 * sigma ** 2)); k = k / k.sum()
        self.radius = radius
        self.register_buffer("kh", k.view(1, 1, 1, -1))
        self.register_buffer("kv", k.view(1, 1, -1, 1))
    def forward(self, x):
        r = self.radius
        x = F.conv2d(F.pad(x, (r, r, 0, 0), mode="replicate"), self.kh)
        return F.conv2d(F.pad(x, (0, 0, r, r), mode="replicate"), self.kv)

def imagenet_weights(enum):
    return None if SMOKE else enum.IMAGENET1K_V1

class ResNetPyramid(nn.Module):
    # Returns layer1, layer2, layer3 feature maps of a torchvision ResNet.
    def __init__(self, net):
        super().__init__()
        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool)
        self.layer1, self.layer2, self.layer3 = net.layer1, net.layer2, net.layer3
    def forward(self, x):
        f1 = self.layer1(self.stem(x)); f2 = self.layer2(f1); f3 = self.layer3(f2)
        return [f1, f2, f3]

class STFPM(nn.Module):
    # Student-Teacher Feature Pyramid Matching (Wang et al., 2021).
    def __init__(self, img_size=IMG_SIZE):
        super().__init__()
        self.img_size = img_size
        self.norm = Normalize()
        self.teacher = ResNetPyramid(models.resnet18(weights=imagenet_weights(models.ResNet18_Weights)))
        self.student = ResNetPyramid(models.resnet18(weights=None))
        for p in self.teacher.parameters():
            p.requires_grad_(False)
        self.blur = GaussianBlur(4.0)

    def train(self, mode=True):
        super().train(mode); self.teacher.eval()   # teacher BN statistics stay frozen
        return self

    def pyramids(self, x):
        x = self.norm(x)
        with torch.no_grad():
            t = self.teacher(x)
        return t, self.student(x)

    @staticmethod
    def loss(t_feats, s_feats):
        return sum(0.5 * (F.normalize(t, dim=1) - F.normalize(s, dim=1)).pow(2).sum(1).mean()
                   for t, s in zip(t_feats, s_feats))

    def forward(self, x):
        # Returns (anomaly map [B,1,H,W], image score [B])
        t_feats, s_feats = self.pyramids(x)
        amap = None
        for t, s in zip(t_feats, s_feats):
            m = 0.5 * (F.normalize(t, dim=1) - F.normalize(s, dim=1)).pow(2).sum(1, keepdim=True)
            m = F.interpolate(m, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False)
            amap = m if amap is None else amap * m
        amap = self.blur(amap)
        return amap, topk_score(amap)

In [ ]:
class PatchCore(nn.Module):
    # PatchCore (Roth et al., 2022): locally aware patch features + greedy coreset memory bank.
    def __init__(self, img_size=IMG_SIZE, coreset_ratio=PATCHCORE_RATIO):
        super().__init__()
        net = (models.resnet18(weights=None) if SMOKE else
               models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1))
        self.backbone = ResNetPyramid(net).eval()
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.norm, self.blur = Normalize(), GaussianBlur(4.0)
        self.img_size, self.ratio = img_size, coreset_ratio
        self.register_buffer("bank", torch.empty(0))

    @torch.no_grad()
    def embed(self, x):
        _, f2, f3 = self.backbone(self.norm(x))
        f2 = F.avg_pool2d(f2, 3, 1, 1)
        f3 = F.interpolate(F.avg_pool2d(f3, 3, 1, 1), size=f2.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([f2, f3], 1)                       # [B, C, h, w]

    @torch.no_grad()
    def fit(self, images, batch=16):
        feats = []
        for i in range(0, len(images), batch):
            e = self.embed(images[i:i + batch].to(DEVICE))
            feats.append(e.permute(0, 2, 3, 1).reshape(-1, e.shape[1]))
        feats = torch.cat(feats)
        idx = greedy_coreset(feats, self.ratio)
        self.bank = feats[idx].contiguous()
        return feats.shape[0], self.bank.shape[0]

    @torch.no_grad()
    def forward(self, x):
        e = self.embed(x)
        b, c, h, w = e.shape
        q = e.permute(0, 2, 3, 1).reshape(-1, c)
        d = torch.cat([torch.cdist(q[i:i + 4096], self.bank).amin(1) for i in range(0, len(q), 4096)])
        m = d.view(b, 1, h, w)
        m = self.blur(F.interpolate(m, size=(self.img_size, self.img_size), mode="bilinear", align_corners=False))
        return m, topk_score(m)

@torch.no_grad()
def greedy_coreset(x, ratio, proj_dim=128):
    # Greedy k-center selection on a random projection (keeps the GPU busy, no host syncs in the loop).
    n = x.shape[0]; k = max(1, int(n * ratio))
    g = torch.Generator(device=x.device).manual_seed(SEED)
    z = x @ (torch.randn(x.shape[1], proj_dim, device=x.device, generator=g) / math.sqrt(proj_dim))
    sel = torch.empty(k, dtype=torch.long, device=x.device)
    sel[0] = 0
    min_d = (z - z[0]).pow(2).sum(1)
    for j in range(1, k):
        i = torch.argmax(min_d)
        sel[j] = i
        min_d = torch.minimum(min_d, (z - z[i]).pow(2).sum(1))
    return sel

## 4. Training, calibration and evaluation helpers

In [ ]:
@torch.no_grad()
def holdout_loss(model, x, batch=16):
    model.eval()
    total = 0.0
    for i in range(0, len(x), batch):
        t, s = model.pyramids(x[i:i + batch].to(DEVICE))
        total += model.loss(t, s).item() * len(x[i:i + batch])
    model.train()
    return total / len(x)

def train_stfpm(train_x, holdout_x, log_every=10):
    # Stabilised STFPM training: gradient clipping, cosine LR decay, non-finite-loss guard, and the
    # student checkpoint with the lowest loss on held-out *defect-free* training images is kept.
    model = STFPM().to(DEVICE).train()
    opt = torch.optim.SGD(model.student.parameters(), lr=STFPM_LR, momentum=0.9, weight_decay=1e-4)
    steps = STFPM_EPOCHS * math.ceil(len(train_x) / STFPM_BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=STFPM_LR, total_steps=steps, pct_start=0.05)
    history, n, best, best_state, skipped = [], len(train_x), float("inf"), None, 0
    for epoch in range(1, STFPM_EPOCHS + 1):
        perm, total = torch.randperm(n), 0.0
        for i in range(0, n, STFPM_BATCH):
            x = train_x[perm[i:i + STFPM_BATCH]].to(DEVICE, non_blocking=True)
            if random.random() < 0.5: x = x.flip(3)        # textures are flip-invariant
            if random.random() < 0.5: x = x.flip(2)
            t, s = model.pyramids(x)
            loss = model.loss(t, s)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1; sched.step(); continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.student.parameters(), 1.0)
            opt.step(); sched.step()
            total += loss.item() * len(x)
        history.append(total / n)
        h = holdout_loss(model, holdout_x)
        if h < best:
            best, best_state = h, {k: v.detach().clone() for k, v in model.student.state_dict().items()}
        if epoch % log_every == 0 or epoch == 1:
            print(f"  epoch {epoch:3d}/{STFPM_EPOCHS}  train {history[-1]:.4f}  holdout {h:.4f}  best {best:.4f}")
    model.student.load_state_dict(best_state)
    if skipped: print(f"  skipped {skipped} non-finite steps")
    return model.eval(), history, best

@torch.no_grad()
def predict(model, images, batch=16):
    maps, scores = [], []
    for i in range(0, len(images), batch):
        m, s = model(images[i:i + batch].to(DEVICE))
        maps.append(m[:, 0].float().cpu()); scores.append(s.float().cpu())
    return torch.cat(maps).numpy(), torch.cat(scores).numpy()

def calibrate(val_maps, val_scores):
    # Everything here comes from defect-free images only.
    mu, sd = float(val_scores.mean()), float(val_scores.std() + 1e-8)
    return {"threshold": mu + THRESH_SIGMAS * sd, "val_mean": mu, "val_std": sd,
            "pixel_p50": float(np.median(val_maps)), "pixel_max": float(np.percentile(val_maps, 99.9))}

def evaluate(scores, maps, test, masks, calib):
    labels = np.array([t["label"] for t in test])
    thr = calib["threshold"]
    pred = scores > thr
    p, r, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
    pr, rc, _ = precision_recall_curve(labels, scores)
    best_f1 = float(np.max(2 * pr * rc / np.clip(pr + rc, 1e-8, None)))
    per_type = {}
    for d in sorted({t["defect"] for t in test}):
        idx = [i for i, t in enumerate(test) if t["defect"] == d]
        per_type[d] = float(pred[idx].mean()) if d != "good" else float(1 - pred[idx].mean())
    return {
        "image_auroc": float(roc_auc_score(labels, scores)),
        "pixel_auroc": float(roc_auc_score(masks.reshape(-1), maps.reshape(-1))),
        "precision": float(p), "recall": float(r), "f1": float(f1),
        "false_alarm_rate": float(pred[labels == 0].mean()),
        "oracle_best_f1": best_f1,                      # uses test labels, reported for reference only
        "per_defect_detection": per_type,               # "good" = fraction correctly passed
    }

def gpu_latency_ms(model, runs=50):
    x = torch.rand(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    with torch.no_grad():
        for _ in range(5): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs): model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000 / runs

## 5. ONNX export
The deployed STFPM model is exported to ONNX and checked against PyTorch, then timed on the CPU with ONNX Runtime, which is the same runtime the demo app uses.

In [ ]:
import onnxruntime as ort

def export_onnx(model, path):
    model = model.eval().cpu()
    dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
    kwargs = dict(input_names=["image"], output_names=["anomaly_map", "score"],
                  dynamic_axes={"image": {0: "batch"}, "anomaly_map": {0: "batch"}, "score": {0: "batch"}},
                  opset_version=17)
    try:
        torch.onnx.export(model, (dummy,), path, dynamo=False, **kwargs)
    except TypeError:                        # older torch without the dynamo flag
        torch.onnx.export(model, (dummy,), path, **kwargs)
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    with torch.no_grad():
        ref_map, ref_score = model(dummy)
    out_map, out_score = sess.run(None, {"image": dummy.numpy()})
    rel_err = float(np.abs(out_score - ref_score.numpy()).max() / (np.abs(ref_score.numpy()).max() + 1e-12))
    model.to(DEVICE)
    return sess, rel_err

def onnx_cpu_latency_ms(sess, runs=50):
    x = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    for _ in range(5): sess.run(None, {"image": x})
    times = []
    for _ in range(runs):
        t0 = time.perf_counter(); sess.run(None, {"image": x}); times.append(time.perf_counter() - t0)
    return float(np.median(times) * 1000)

@torch.no_grad()
def torch_cpu_latency_ms(model, runs=5):
    model = model.eval().cpu()
    x = torch.rand(1, 3, IMG_SIZE, IMG_SIZE)
    model(x)
    t0 = time.perf_counter()
    for _ in range(runs): model(x)
    ms = (time.perf_counter() - t0) * 1000 / runs
    model.to(DEVICE)
    return ms

## 6. Figures

In [ ]:
CMAP = matplotlib.colormaps["inferno"]

def overlay(img_chw, amap, calib):
    # Same rule as the demo app: heat starts at the typical normal level, saturates at 1.5x the threshold.
    img = img_chw.permute(1, 2, 0).numpy()
    lo, hi = calib["pixel_p50"], 1.5 * calib["threshold"]
    v = np.clip((amap - lo) / (hi - lo + 1e-12), 0, 1)
    a = (0.85 * v)[..., None]
    return np.clip((1 - a) * img + a * CMAP(v)[..., :3], 0, 1)

def plot_category(cat, test, test_x, masks, res):
    labels = np.array([t["label"] for t in test])
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for ax, name in zip(axes[:2], ["stfpm", "patchcore"]):
        s, thr = res[name]["scores"], res[name]["calib"]["threshold"]
        bins = np.linspace(min(s.min(), thr), max(s.max(), thr), 40)
        ax.hist(s[labels == 0], bins, alpha=.7, label="defect-free", color="#5FAF8F")
        ax.hist(s[labels == 1], bins, alpha=.7, label="defective", color="#C8473B")
        ax.axvline(thr, color="k", ls="--", label="threshold (normal-only)")
        ax.set_title(f"{cat} · {name} score distribution"); ax.legend()
    for name, color in [("stfpm", "#2E4A8B"), ("patchcore", "#E0A526")]:
        fpr, tpr, _ = roc_curve(labels, res[name]["scores"])
        axes[2].plot(fpr, tpr, color=color, label=f"{name} (AUROC {res[name]['metrics']['image_auroc']:.3f})")
    axes[2].plot([0, 1], [0, 1], "k:"); axes[2].set_title(f"{cat} · image-level ROC"); axes[2].legend()
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "figures", f"{cat}_scores.png"), dpi=120); plt.show()

    picks = []
    for d in sorted({t["defect"] for t in test} - {"good"}):
        picks.append(next(i for i, t in enumerate(test) if t["defect"] == d))
    picks = picks[:6]
    fig, axes = plt.subplots(4, len(picks), figsize=(2.6 * len(picks), 10.8), squeeze=False)
    for col, i in enumerate(picks):
        axes[0, col].imshow(test_x[i].permute(1, 2, 0)); axes[0, col].set_title(test[i]["defect"])
        axes[1, col].imshow(masks[i], cmap="gray")
        axes[2, col].imshow(overlay(test_x[i], res["stfpm"]["maps"][i], res["stfpm"]["calib"]))
        axes[3, col].imshow(overlay(test_x[i], res["patchcore"]["maps"][i], res["patchcore"]["calib"]))
    for row, name in enumerate(["input", "ground truth", "STFPM", "PatchCore"]):
        axes[row, 0].set_ylabel(name, fontsize=12)
    for ax in axes.ravel(): ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "figures", f"{cat}_heatmaps.png"), dpi=110); plt.show()

def save_samples(cat, test, n_good=2):
    # A few test images for the demo app's sample strip (resized to keep the repo light).
    dest = os.path.join(OUT, "samples", cat); os.makedirs(dest, exist_ok=True)
    chosen = [t for t in test if t["defect"] == "good"][:n_good]
    for d in sorted({t["defect"] for t in test} - {"good"}):
        chosen.append(next(t for t in test if t["defect"] == d))
    for t in chosen:
        stem = os.path.splitext(os.path.basename(t["path"]))[0]
        Image.open(t["path"]).convert("RGB").resize((512, 512), Image.BILINEAR) \
             .save(os.path.join(dest, f"{t['defect']}_{stem}.jpg"), quality=90)

## 7. Run every category
For each category: train STFPM, fit PatchCore, calibrate both on defect-free images, evaluate on the test set, export STFPM to ONNX and time everything.

**Calibration set.** Held-out *training* images turned out to score lower than the test-time good images (they were photographed in the same session as the training data), which put the threshold too low and caused many false alarms. A factory would calibrate on a few good samples captured on the actual line at deployment. We mirror that: half of the test-set *good* images are used for calibration only and removed from evaluation. No defect labels are used.

In [ ]:
RESULTS = {}
for cat in CATEGORIES:
    print(f"\n=== {cat} ===")
    seed_everything(SEED)
    train_p, val_p, test = build_split(cat)
    holdout_p = val_p                                   # defect-free training images, used for checkpoint selection
    if CALIB_ON_DEPLOYMENT_NORMALS:
        good_idx = [i for i, t in enumerate(test) if t["label"] == 0]
        random.Random(SEED).shuffle(good_idx)
        calib_idx = set(good_idx[:len(good_idx) // 2])
        val_p = [test[i]["path"] for i in sorted(calib_idx)]
        test = [t for i, t in enumerate(test) if i not in calib_idx]
    train_x, val_x, holdout_x = load_stack(train_p), load_stack(val_p), load_stack(holdout_p)
    test_x = load_stack([t["path"] for t in test])
    masks = np.stack([load_mask(t["mask"]) for t in test])
    res = {}

    t0 = time.time()
    runs = []
    for attempt in range(STFPM_RESTARTS):
        seed_everything(SEED + attempt)
        print(f"  STFPM run {attempt + 1}/{STFPM_RESTARTS}")
        runs.append(train_stfpm(train_x, holdout_x))
    best_i = min(range(len(runs)), key=lambda i: runs[i][2])
    stfpm, hist, _ = runs[best_i]
    print(f"  kept run {best_i + 1} (held-out losses: {', '.join(f'{r[2]:.4f}' for r in runs)})")
    del runs
    stfpm_train_s = time.time() - t0

    t0 = time.time()
    pc = PatchCore().to(DEVICE)
    n_patches, n_bank = pc.fit(train_x)
    pc_fit_s = time.time() - t0
    print(f"  PatchCore memory bank: {n_bank:,} of {n_patches:,} patches")

    for name, model, fit_s in [("stfpm", stfpm, stfpm_train_s), ("patchcore", pc, pc_fit_s)]:
        val_maps, val_scores = predict(model, val_x)
        calib = calibrate(val_maps, val_scores)
        maps, scores = predict(model, test_x)
        metrics = evaluate(scores, maps, test, masks, calib)
        metrics["fit_seconds"] = round(fit_s, 1)
        metrics["gpu_latency_ms"] = gpu_latency_ms(model)
        res[name] = {"maps": maps, "scores": scores, "calib": calib, "metrics": metrics}
        print(f"  {name:9s} image AUROC {metrics['image_auroc']:.4f} | pixel AUROC {metrics['pixel_auroc']:.4f} "
              f"| F1 {metrics['f1']:.3f} | false alarms {metrics['false_alarm_rate']:.1%}")

    # CPU comparison: PatchCore in PyTorch vs STFPM in ONNX Runtime
    res["patchcore"]["metrics"]["cpu_latency_ms"] = torch_cpu_latency_ms(pc)
    mdir = os.path.join(OUT, "models", cat); os.makedirs(mdir, exist_ok=True)
    onnx_path = os.path.join(mdir, "stfpm.onnx")
    sess, rel_err = export_onnx(stfpm, onnx_path)
    res["stfpm"]["metrics"]["cpu_latency_ms"] = onnx_cpu_latency_ms(sess)
    res["stfpm"]["metrics"]["onnx_size_mb"] = round(os.path.getsize(onnx_path) / 2**20, 1)
    res["stfpm"]["metrics"]["onnx_rel_error"] = rel_err
    print(f"  CPU latency: STFPM/ONNX {res['stfpm']['metrics']['cpu_latency_ms']:.1f} ms vs "
          f"PatchCore/PyTorch {res['patchcore']['metrics']['cpu_latency_ms']:.1f} ms | ONNX rel. error {rel_err:.2e}")

    meta = {"category": cat, "model": "stfpm-resnet18", "img_size": IMG_SIZE, "input": "RGB float32 in [0,1], NCHW",
            **res["stfpm"]["calib"], "metrics": res["stfpm"]["metrics"], "trained_on": "MVTec AD (CC BY-NC-SA 4.0)"}
    with open(os.path.join(mdir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    plot_category(cat, test, test_x, masks, res)
    save_samples(cat, test)
    RESULTS[cat] = {"n_train": len(train_p), "n_val": len(val_p), "n_test": len(test),
                    "n_defective": int(sum(t["label"] for t in test)), "stfpm_loss_curve": hist,
                    **{name: res[name]["metrics"] for name in ["stfpm", "patchcore"]}}
    del stfpm, pc, res, train_x, val_x, holdout_x, test_x, masks
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

## 8. Summary, results file and resume numbers

In [ ]:
def mean_of(model, key):
    return float(np.mean([RESULTS[c][model][key] for c in RESULTS]))

rows = ["| Category | Model | Image AUROC | Pixel AUROC | F1 (normal-only threshold) | False alarms | GPU ms | CPU ms |",
        "|---|---|---|---|---|---|---|---|"]
for c in RESULTS:
    for name, label in [("stfpm", "STFPM (deployed, ONNX)"), ("patchcore", "PatchCore")]:
        m = RESULTS[c][name]
        rows.append(f"| {c} | {label} | {m['image_auroc']:.3f} | {m['pixel_auroc']:.3f} | {m['f1']:.3f} | "
                    f"{m['false_alarm_rate']:.1%} | {m['gpu_latency_ms']:.1f} | {m['cpu_latency_ms']:.1f} |")
for name, label in [("stfpm", "STFPM (deployed, ONNX)"), ("patchcore", "PatchCore")]:
    rows.append(f"| **mean** | {label} | {mean_of(name, 'image_auroc'):.3f} | {mean_of(name, 'pixel_auroc'):.3f} | "
                f"{mean_of(name, 'f1'):.3f} | {mean_of(name, 'false_alarm_rate'):.1%} | "
                f"{mean_of(name, 'gpu_latency_ms'):.1f} | {mean_of(name, 'cpu_latency_ms'):.1f} |")
table = "\n".join(rows)
print(table)

summary = {
    "categories": list(RESULTS), "deploy_category": DEPLOY_CATEGORY, "img_size": IMG_SIZE,
    "stfpm_epochs": STFPM_EPOCHS, "threshold_rule": (f"mean + {THRESH_SIGMAS} std of " + ("deployment-time defect-free images (half of the test-set good images, excluded from evaluation)"
                       if CALIB_ON_DEPLOYMENT_NORMALS else "held-out defect-free training images")),
    "image_score": f"mean of top {TOPK_FRACTION:.0%} pixels",
    "environment": {"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
                    "cpu": platform.processor() or platform.machine(), "cpu_threads": os.cpu_count()},
    "results": RESULTS,
}
with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(summary, f, indent=2)
with open(os.path.join(OUT, "RESULTS.md"), "w") as f:
    f.write(table + "\n")

d = RESULTS.get(DEPLOY_CATEGORY, next(iter(RESULTS.values())))["stfpm"]
speedup = mean_of("patchcore", "cpu_latency_ms") / mean_of("stfpm", "cpu_latency_ms")
print(f'''
Resume bullets (numbers from this run):
- Built LoomGuard, an unsupervised fabric-defect detector trained only on defect-free images; reached
  {mean_of("stfpm", "image_auroc"):.1%} mean image AUROC and {mean_of("stfpm", "pixel_auroc"):.1%} pixel AUROC across {len(RESULTS)} MVTec AD texture categories.
- Benchmarked a student-teacher model (STFPM) against PatchCore; deployed STFPM as a {d["onnx_size_mb"]} MB ONNX model
  running in {mean_of("stfpm", "cpu_latency_ms"):.0f} ms/image on CPU, {speedup:.0f}x faster than PatchCore on the same CPU.
- Calibrated decision thresholds from defect-free data only (no defect labels), giving {d["f1"]:.2f} F1 with a
  {d["false_alarm_rate"]:.0%} false-alarm rate on {DEPLOY_CATEGORY}; served via FastAPI with a live heatmap dashboard.
''')

In [ ]:
bad = [c for c in RESULTS if RESULTS[c]["stfpm"]["pixel_auroc"] < 0.9]
print("HEALTH CHECK:", "all categories trained correctly." if not bad else f"weak STFPM result on {bad}.")

archive = shutil.make_archive(OUT + "_artifacts", "zip", OUT)
print("Download from the Output panel:", archive, f"({os.path.getsize(archive) / 2**20:.1f} MB)")

## Next steps
1. Download `loomguard_artifacts.zip` from the **Output** panel on the right.
2. Unzip it into the repo root, so you get `models/`, `samples/`, `figures/`, `results.json` and `RESULTS.md`.
3. Run `python tools/fill_readme.py` to put these numbers into the README, then start the dashboard with `uvicorn app.server:app`.